# synthetic data generation

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Generate or verify the frozen seed-20261023 dataset

In [ ]:
# Synthetic developmental crescent generator, original frozen cortical v1.1 DGP
# Run in original B mode: no tangential off-axis gene-expression perturbation.
# Biological geometry, expression mechanisms, flow, sampling and random section poses are unchanged.
import os, json, hashlib
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd

# ==================== FROZEN V1.1 GENERATING MODEL ====================
VERSION = "cortical_crescent_v1.1_positional_20261008"


@dataclass(frozen=True)
class Config:
    n_tissues: int = 10
    cells_at_t0: int = 1800
    n_genes: int = 48
    population: str = "growing"  # 'growing' or 'constant'
    sampling: str = "uniform_physical"  # or 'uniform_intrinsic'
    regime: str = "B"  # A=pure, B=orthogonal, C=parallel, D=off-axis
    parallel_strength: float = 0.0
    offaxis_strength: float = 0.0
    library_log_sd: float = 0.0
    seed: int = 20261008

    def validate(self):
        if self.n_tissues != 10:
            raise ValueError("Frozen cortical geometry has exactly ten tissues")
        if self.n_genes != 48:
            raise ValueError("Cortical v1.1 has exactly 48 genes")
        if self.cells_at_t0 < 1:
            raise ValueError("cells_at_t0 must be positive")
        if self.population not in ("growing", "constant"):
            raise ValueError("population must be growing or constant")
        if self.sampling not in ("uniform_physical", "uniform_intrinsic"):
            raise ValueError("unknown sampling mode")
        if self.regime not in ("A", "B", "C", "D"):
            raise ValueError("unknown expression regime")
        if min(self.parallel_strength, self.offaxis_strength, self.library_log_sd) < 0:
            raise ValueError("perturbation strengths and library_log_sd must be nonnegative")
        if self.regime != "C" and self.parallel_strength:
            raise ValueError("parallel_strength is only valid in regime C")
        if self.regime != "D" and self.offaxis_strength:
            raise ValueError("offaxis_strength is only valid in regime D")


# Frozen geometry and migration settings from the notebook.
LENGTH = 1.25
THICKNESS0 = 0.17
CURVATURE_MAX = 3.35
GROWTH_EDGE = 0.82
GROWTH_CENTER = 0.70
GROWTH_WIDTH = 0.26
ROTATIONS = {}  # No artificial rotations. Preserve translations and biology.
VELOCITY_SCALE = 1.8
VELOCITY_ASYMMETRY = 0.45
VELOCITY_FINAL_FRACTION = 0.10


def curvature(t):
    return CURVATURE_MAX * np.asarray(t)**1.15


def thickness(u, t):
    s = np.asarray(u) - .5
    bump = np.exp(-.5 * (s / GROWTH_WIDTH)**2)
    return THICKNESS0 * (1 + np.asarray(t) * (GROWTH_EDGE + GROWTH_CENTER*bump))


def area_jacobian(u, z, t):
    h = thickness(u, t)
    return h * (LENGTH + curvature(t) * np.asarray(z) * h)


def tissue_area(t):
    u = np.linspace(0, 1, 2049)
    h = thickness(u, t)
    return float(np.sum(.5*(LENGTH*h[:-1]+.5*curvature(t)*h[:-1]**2 + LENGTH*h[1:]+.5*curvature(t)*h[1:]**2)*np.diff(u)))


def geometry(u, z, t):
    """G_t(u,z) and dG/dz; z increases along outward cortical normal."""
    u, z = np.broadcast_arrays(np.asarray(u, float), np.asarray(z, float))
    s = u - .5
    k = curvature(t)
    theta = k*s
    cx = LENGTH*s*np.sinc(theta/np.pi)
    cy = .5*LENGTH*k*s*s*np.sinc(theta/(2*np.pi))**2
    h = thickness(u, t)
    nx, ny = np.sin(theta), -np.cos(theta)
    return cx + z*h*nx, cy + z*h*ny, h*nx, h*ny


def rotate(x, y, angle):
    a = np.deg2rad(angle)
    return np.cos(a)*x - np.sin(a)*y, np.sin(a)*x + np.cos(a)*y


def v_z(z, t):
    z, t = np.broadcast_arrays(np.asarray(z, float), np.asarray(t, float))
    spatial = VELOCITY_SCALE*z*(1-z)*(1+VELOCITY_ASYMMETRY*(2*z-1))
    temporal = 1 - (1-VELOCITY_FINAL_FRACTION)*(3*t*t-2*t*t*t)
    return spatial*temporal


def v_z_derivative(z, t):
    z = np.asarray(z, float)
    f = (1-2*z)*(1+VELOCITY_ASYMMETRY*(2*z-1)) + 2*VELOCITY_ASYMMETRY*z*(1-z)
    temporal = 1-(1-VELOCITY_FINAL_FRACTION)*(3*t*t-2*t*t*t)
    return VELOCITY_SCALE*f*temporal


def flow(a, t_end, steps_per_unit=300):
    """RK4 physical migration F_t(a); keeps [0,1] endpoints fixed."""
    z = np.asarray(a, float).copy()
    n = max(1, int(np.ceil(steps_per_unit*t_end)))
    dt = t_end/n
    for i in range(n):
        t = i*dt
        k1 = v_z(z, t)
        k2 = v_z(z + .5*dt*k1, t + .5*dt)
        k3 = v_z(z + .5*dt*k2, t + .5*dt)
        k4 = v_z(z + dt*k3, t + dt)
        z += dt*(k1 + 2*k2 + 2*k3 + k4)/6
    return z


def invert_flow(z, t):
    ag = np.linspace(0, 1, 4001)
    zg = flow(ag, t)
    return np.interp(z, zg, ag)


def sample_intrinsic(n, t, rng, sampling):
    if sampling == "uniform_intrinsic":
        return rng.uniform(size=n), rng.uniform(size=n)
    # Rejection sampling with the exact area Jacobian, so cell positions
    # are uniformly distributed in the *physical* two-dimensional tissue.
    max_j = float(np.max(area_jacobian(np.linspace(0,1,513), 1., t)))
    us, zs = [], []
    remaining = n
    while remaining:
        batch = max(64, int(remaining*1.7))
        u = rng.uniform(size=batch)
        z = rng.uniform(size=batch)
        keep = rng.uniform(size=batch)*max_j <= area_jacobian(u,z,t)
        x, y = u[keep][:remaining], z[keep][:remaining]
        us.append(x); zs.append(y)
        remaining -= len(x)
    return np.concatenate(us), np.concatenate(zs)


def _expression_parameters(seed):
    """Fixed family-correlated gene directions with mutually orthogonal axes."""
    rng = np.random.default_rng(seed + 9001)
    raw = np.zeros((48,8))
    prototypes = 3*np.eye(8) + .35*rng.normal(size=(8,8))
    for family in range(8):
        raw[family*5:(family+1)*5] = prototypes[family] + .09*rng.normal(size=(5,8))
    loading, _ = np.linalg.qr(raw, mode="reduced")
    for j in range(8):
        if loading[j*5:(j+1)*5,j].mean() < 0:
            loading[:,j] *= -1
    baseline = 6.7 + rng.normal(0,.32,48)
    # Baselines ensure Psi>0 over all four regimes for nominal stress strengths.
    baseline += 2.0
    offaxis_loading = rng.normal(size=48)
    offaxis_loading /= np.linalg.norm(offaxis_loading)
    return loading, baseline, offaxis_loading


AMP = np.array([3.5,3.1,3.3,2.0,3.6,3.6,2.8,2.8])


def spatial_basis(a):
    early = 1-2*a
    mid = 2*np.exp(-.5*((a-.43)/.15)**2)-.65
    late = 2/(1+np.exp(-(a-.64)/.105))-1
    band = np.sin(2*np.pi*a)
    return np.stack([early,mid,late,band], axis=-1)


def spatial_basis_derivative(a):
    gaussian = np.exp(-.5*((a-.43)/.15)**2)
    logistic = 1/(1+np.exp(-(a-.64)/.105))
    return np.stack([
        -2*np.ones_like(a),
        -2*gaussian*(a-.43)/(.15**2),
        2*logistic*(1-logistic)/.105,
        2*np.pi*np.cos(2*np.pi*a)
    ], axis=-1)


# This is the only v1.1 change to the frozen v1.0 DGP.
# Time rotation in two orthogonal expression-space planes. The amplitudes
# depend on initial material position a, NOT current z; consequently
# q = partial_t Psi|a is orthogonal to partial_a Psi for every a,t in B.
OMEGA1, OMEGA2 = 1.5, 1.9
SECOND_PAIR_WEIGHT = 0.40
DOMINANCE_TRANSITION_A0 = 0.85
MOLECULAR_SPATIAL_POWER = 3.0

def calibration_amplitude():
    a = DOMINANCE_TRANSITION_A0
    # Choose amplitude such that ||q|| / (v ||Psi_a||) = 1 at t=0,a=.85.
    # Orthogonal gene-space groups mean the loading matrix cancels.
    base_sq = float(np.sum((spatial_basis_derivative(np.array([a]))[0]*AMP[:4])**2))
    v = float(v_z(a,0))
    qper2 = (a**3*OMEGA1)**2 + (SECOND_PAIR_WEIGHT*a**4*OMEGA2)**2
    aper2 = (3*a*a)**2 + (SECOND_PAIR_WEIGHT*4*a**3)**2
    den = qper2-v*v*aper2
    if den<=0: raise RuntimeError("Cannot calibrate initial dominance boundary")
    return float(v*np.sqrt(base_sq/den))

MOLECULAR_AMPLITUDE = calibration_amplitude()

def molecular_speed_material(a):
    a=np.asarray(a,float)
    return MOLECULAR_AMPLITUDE*np.sqrt((OMEGA1*a**3)**2+
             (SECOND_PAIR_WEIGHT*OMEGA2*a**4)**2)

def molecular_rho_material(a,t):
    return np.asarray(t)*molecular_speed_material(a)

def psi(a, u, t, cfg, params):
    """Psi(a,u,t)=2 sqrt(expression rate); positive, spatial q in regime B."""
    a,u,t = np.broadcast_arrays(np.asarray(a,float),np.asarray(u,float),np.asarray(t,float))
    loading, baseline, offaxis_loading = params
    if cfg.regime=='A':
        molecular=np.zeros(a.shape+(4,))
    else:
        A1=MOLECULAR_AMPLITUDE*a**3
        A2=MOLECULAR_AMPLITUDE*SECOND_PAIR_WEIGHT*a**4
        molecular=np.stack([A1*np.cos(OMEGA1*t),A1*np.sin(OMEGA1*t),
                            A2*np.cos(OMEGA2*t),A2*np.sin(OMEGA2*t)],axis=-1)
    spatial_terms=spatial_basis(a)*AMP[:4]
    out=baseline + np.concatenate([spatial_terms,molecular],axis=-1)@loading.T
    if cfg.regime=='C' and cfg.parallel_strength:
        out+=cfg.parallel_strength*.15*t[...,None]*(
            (spatial_basis_derivative(a)*AMP[:4])@loading[:,:4].T)
    if cfg.regime=='D' and cfg.offaxis_strength:
        profile=np.sin(2*np.pi*u)*np.cos(np.pi*a)*(1+.35*t)
        out+=cfg.offaxis_strength*3*profile[...,None]*offaxis_loading
    if not np.isfinite(out).all() or np.min(out)<=0:
        raise ValueError('Nonpositive Psi: increase constant baseline for specified regime')
    return out

def stabilized_params(cfg):
    load,base,off=_expression_parameters(cfg.seed)
    a=np.linspace(0,1,501);t=np.linspace(0,1,33)
    aa,tt=np.meshgrid(a,t)
    # compute finite domain minimum; positivity correction is gene-constant
    # and therefore does NOT change derivatives or Fisher orthogonality.
    space=spatial_basis(aa)*AMP[:4]
    A1=MOLECULAR_AMPLITUDE*aa**3
    A2=MOLECULAR_AMPLITUDE*SECOND_PAIR_WEIGHT*aa**4
    m=np.stack([A1*np.cos(OMEGA1*tt),A1*np.sin(OMEGA1*tt),
                A2*np.cos(OMEGA2*tt),A2*np.sin(OMEGA2*tt)],axis=-1)
    vals=np.concatenate([space,m],axis=-1)@load.T
    base=base+np.maximum(1.5-(base+vals).min(axis=(0,1)),0.)
    return (load,base,off)

def expression_rate(a,u,t,cfg,params=None):
    if params is None:
        params = stabilized_params(cfg)
    return .25*psi(a,u,t,cfg,params)**2


def _material_derivatives(a,u,t,cfg,params):
    h = 2e-5
    da = (psi(a+h,u,t,cfg,params)-psi(a-h,u,t,cfg,params))/(2*h)
    dt = (psi(a,u,t+h,cfg,params)-psi(a,u,t-h,cfg,params))/(2*h)
    # The flow Jacobian converts derivatives with respect to a into z.
    fa = (flow(a+h,t)-flow(a-h,t))/(2*h)
    dz = da/fa[:,None]
    fisher = v_z(flow(a,t),t) - np.einsum('ng,ng->n',dt,dz)/np.maximum(np.einsum('ng,ng->n',dz,dz),1e-12)
    return dt, dz, fisher


def _birth_rate_constant_density(u,z,t):
    """Continuity-equation source per cell for uniform physical density.

    gamma = [partial_t J + partial_z(J*v)]/J. This is the net
    proliferation/death rate needed by the specified evolving density,
    geometry and migration velocity; not an independently simulated lineage.
    """
    h = thickness(u,t)
    s = np.asarray(u)-.5
    ht = THICKNESS0*(GROWTH_EDGE+GROWTH_CENTER*np.exp(-.5*(s/GROWTH_WIDTH)**2))
    k, kt = curvature(t), CURVATURE_MAX*1.15*t**.15
    J = area_jacobian(u,z,t)
    Jt = ht*(LENGTH+2*k*z*h)+kt*z*h*h
    Jz = k*h*h
    return (Jt+Jz*v_z(z,t))/J + v_z_derivative(z,t)


def generate(cfg: Config = Config()):
    cfg.validate()
    params = stabilized_params(cfg)
    times = np.linspace(0,1,cfg.n_tissues)
    areas = np.array([tissue_area(float(t)) for t in times])
    records, truth, counts, rates, q_parts, dz_parts = [],[],[],[],[],[]
    n0 = cfg.cells_at_t0
    for i,t in enumerate(times):
        n = n0 if cfg.population == "constant" else int(np.round(n0*areas[i]/areas[0]))
        rng_cells = np.random.default_rng(cfg.seed+100+i)
        rng_counts = np.random.default_rng(cfg.seed+10000+i)
        rng_pose = np.random.default_rng(cfg.seed+20000+i)
        u,z = sample_intrinsic(n,float(t),rng_cells,cfg.sampling)
        a = invert_flow(z,float(t))
        px,py,dzx,dzy = geometry(u,z,t)
        angle = ROTATIONS.get(i,0.)
        tx,ty = rng_pose.uniform(-.12,.12,2)
        x,y = rotate(px,py,angle)
        x,y = x+tx,y+ty
        vz = v_z(z,t)
        vx,vy = rotate(dzx*vz,dzy*vz,angle)
        expected = expression_rate(a,u,t,cfg,params)
        if cfg.library_log_sd:
            L = np.exp(rng_counts.normal(-.5*cfg.library_log_sd**2,cfg.library_log_sd,size=n))
        else:
            L = np.ones(n)
        y_count = rng_counts.poisson(expected*L[:,None]).astype(np.int32)
        cellids = [f"tissue{i:02d}_cell{j:05d}" for j in range(n)]
        records.append(pd.DataFrame(dict(cell_id=cellids,tissue_id=i,time=t,x=x,y=y)))
        q,dpsi_z,vf = _material_derivatives(a,u,float(t),cfg,params)
        eps = 1e-5
        left, right = max(0.,float(t)-eps), min(1.,float(t)+eps)
        A_prime = (tissue_area(right)-tissue_area(left))/(right-left)
        if cfg.sampling == "uniform_physical":
            gamma = _birth_rate_constant_density(u,z,float(t))
            if cfg.population == "constant":
                gamma -= A_prime/tissue_area(float(t))
        else:
            # Uniform intrinsic sampling implies physical density proportional
            # to 1/J. Its local source is d(log N)/dt + partial_z v.
            gamma = v_z_derivative(z,t)
            if cfg.population == "growing":
                gamma += A_prime/tissue_area(float(t))
        truth.append(pd.DataFrame(dict(
            cell_id=cellids,material_a_true=a,u_true=u,z_true=z,
            vx_migration_true=vx,vy_migration_true=vy,vz_true=vz,
            v_fisher_true=vf,
            q_norm_true=np.linalg.norm(q,axis=1),
            fisher_observability_true=np.sum(dpsi_z**2,axis=1),
            growth_rate_continuity_true=gamma,
            true_library_factor=L,rotation_deg=angle,translation_x=tx,translation_y=ty,
            physical_x_true=px,physical_y_true=py,
        )))
        cols=[f"G{g+1:02d}" for g in range(48)]
        counts.append(pd.DataFrame(y_count,columns=cols).assign(cell_id=cellids))
        rates.append(pd.DataFrame(expected,columns=cols).assign(cell_id=cellids))
        q_parts.append(q.astype(np.float32)); dz_parts.append(dpsi_z.astype(np.float32))
    obs = pd.concat(records,ignore_index=True)
    hidden = pd.concat(truth,ignore_index=True)
    counts = pd.concat(counts,ignore_index=True)[['cell_id']+cols]
    rates = pd.concat(rates,ignore_index=True)[['cell_id']+cols]
    return dict(config=cfg,observed=obs,counts=counts,hidden=hidden,rates=rates,
                q_true=np.concatenate(q_parts),psi_z_true=np.concatenate(dz_parts),
                gene_names=cols,areas=areas,times=times)


from datetime import datetime, timezone

NEW_SEED = 20261023
PROBE_SEED = 20261024
TARGET_TIME = 2.5 / 9.0
SOURCE_TIME = 2.0 / 9.0
PROBE_FRACTION = 0.15

# The default points to the user's existing STP project. Set STP_PROJECT if moved.
PROJECT = Path(os.environ.get(
    'STP_PROJECT',
    os.environ["STP_PROJECT"]
)).expanduser()
EXP = PROJECT / 'Experiments/simulation validation/Synthetic benchmarking/experiments'
SMOKE = os.environ.get('CRESCENT_SMOKE', '0') == '1'
if SMOKE:
    # Local reduced-size structural test; never confuse it with the real benchmark.
    ROOT = Path(os.environ.get('CRESCENT_SMOKE_DIR', str(PROJECT / ".crescent_smoke_test")))
    CELLS_AT_T0 = 60
else:
    if not PROJECT.is_dir():
        raise FileNotFoundError(f'STP project not found: {PROJECT}; set STP_PROJECT to its directory')
    ROOT = EXP / 'synthetic_developmental_crescent_v1'
    CELLS_AT_T0 = 1800

CFG = Config(
    n_tissues=10, cells_at_t0=CELLS_AT_T0, n_genes=48,
    population='growing', sampling='uniform_physical',
    regime='B', parallel_strength=0., offaxis_strength=0.,
    library_log_sd=0., seed=NEW_SEED,
)
CFG.validate()
GENE_COLS = [f'G{g+1:02d}' for g in range(48)]
ROOT_PUBLIC = ROOT / 'train_only/public'
ROOT_EVAL = ROOT / 'evaluator_only'
ROOT_QC = ROOT / 'quality_control'
ROOT_MAN = ROOT / 'manifests'
MANIFEST = ROOT_MAN / 'dataset_manifest.json'

# A copy of the exact generative functions is included in this cell's source.
# If desired, archive the original complete cell/notebook alongside this script.
def write_csv_gz(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False,
              compression={'method':'gzip','compresslevel':6,'mtime':0})


def file_hash(path):
    sha = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(2**20), b''):
            sha.update(chunk)
    return sha.hexdigest()


def make_heldout_tissue(cfg, t, params):
    """Independent snapshot at an unseen time, using the frozen v1.1 DGP."""
    n = (cfg.cells_at_t0 if cfg.population == 'constant' else
         int(np.round(cfg.cells_at_t0 * tissue_area(t)/tissue_area(0.))))
    # Disjoint random streams from observed-tissue generation.
    r_cells = np.random.default_rng(cfg.seed + 510000)
    r_counts = np.random.default_rng(cfg.seed + 520000)
    r_pose = np.random.default_rng(cfg.seed + 530000)
    u, z = sample_intrinsic(n, t, r_cells, cfg.sampling)
    a = invert_flow(z, t)
    px, py, dzx, dzy = geometry(u, z, t)
    rotation_deg = 0.0  # The frozen no-rotation crescent variant.
    tx, ty = r_pose.uniform(-0.12, 0.12, size=2)
    x, y = rotate(px, py, rotation_deg)
    x, y = x + tx, y + ty
    lam = expression_rate(a, u, t, cfg, params)
    if cfg.library_log_sd:
        lib = np.exp(r_counts.normal(-0.5*cfg.library_log_sd**2,
                                    cfg.library_log_sd, size=n))
    else:
        lib = np.ones(n)
    y_count = r_counts.poisson(lam*lib[:,None]).astype(np.int32)
    ids = [f'heldout_t2p5_cell{j:05d}' for j in range(n)]
    obs = pd.DataFrame({'cell_id':ids,'tissue_id':[-1]*n,
                        'time':np.full(n,t),'x':x,'y':y})
    truth = pd.DataFrame({'cell_id':ids,'material_a_true':a,'u_true':u,
                          'z_true':z,'physical_x_true':px,'physical_y_true':py,
                          'rotation_deg':np.full(n,rotation_deg),
                          'translation_x':np.full(n,tx),
                          'translation_y':np.full(n,ty),
                          'vx_migration_true':dzx*v_z(z,t),
                          'vy_migration_true':dzy*v_z(z,t),
                          'vz_true':v_z(z,t),
                          'true_library_factor':lib})
    rates = pd.DataFrame(lam,columns=GENE_COLS).assign(cell_id=ids)[['cell_id']+GENE_COLS]
    counts = pd.DataFrame(y_count,columns=GENE_COLS).assign(cell_id=ids)[['cell_id']+GENE_COLS]
    return obs, truth, counts, rates


def save_new_dataset():
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(
            f'Destination exists but has no valid manifest: {ROOT}\n'
            'Refusing to overwrite. Move/archive it deliberately first.'
        )
    ROOT.mkdir(parents=True,exist_ok=True)
    print('Generating ten observed crescent tissues with seed',NEW_SEED,flush=True)
    result = generate(CFG)
    observed, measured = result['observed'], result['counts']
    source_truth, rates = result['hidden'], result['rates']
    assert len(observed) == len(measured) == len(source_truth) == len(rates)
    assert len(set(observed.cell_id)) == len(observed)
    assert np.allclose(np.sort(observed.time.unique()),np.linspace(0,1,10))
    assert not np.any(np.isclose(observed.time,TARGET_TIME))
    assert (measured[GENE_COLS].to_numpy()>=0).all()
    assert (rates[GENE_COLS].to_numpy()>0).all()
    assert (observed.cell_id.to_numpy()==measured.cell_id.to_numpy()).all()
    assert (observed.cell_id.to_numpy()==source_truth.cell_id.to_numpy()).all()

    params = stabilized_params(CFG)
    # Source identity is retained, but future expression is never exposed to models.
    mask = observed.tissue_id.to_numpy()==2
    truth_source = source_truth.loc[mask].reset_index(drop=True)
    future_lam = expression_rate(
        truth_source.material_a_true.to_numpy(float),
        truth_source.u_true.to_numpy(float),TARGET_TIME,CFG,params)
    source_future = pd.DataFrame(future_lam,columns=GENE_COLS)
    source_future.insert(0,'cell_id',truth_source.cell_id.to_numpy())

    probe_rng = np.random.default_rng(PROBE_SEED)
    source_ids = truth_source.cell_id.to_numpy(str)
    probe_ids = np.sort(probe_rng.choice(
        source_ids,size=max(1,int(round(PROBE_FRACTION*len(source_ids)))),replace=False))
    probe_table = pd.DataFrame({'cell_id':probe_ids})

    print('Generating independent intermediate tissue at t=2.5/9',flush=True)
    tgt_obs, tgt_truth, tgt_counts, tgt_rates = make_heldout_tissue(CFG,TARGET_TIME,params)
    assert not set(tgt_obs.cell_id).intersection(set(observed.cell_id))
    assert (tgt_counts[GENE_COLS].to_numpy()>=0).all()
    assert np.isfinite(tgt_rates[GENE_COLS].to_numpy()).all()

    # Public observations (the *only* training data).
    write_csv_gz(observed,ROOT_PUBLIC/'observed_cells.csv.gz')
    write_csv_gz(measured,ROOT_PUBLIC/'counts.csv.gz')
    ROOT_PUBLIC.mkdir(exist_ok=True,parents=True)
    pd.DataFrame({'gene':GENE_COLS}).to_csv(ROOT_PUBLIC/'genes.csv',index=False)

    # All oracle quantities are contained inside evaluator_only.
    write_csv_gz(source_truth,ROOT_EVAL/'observed_hidden/cell_truth.csv.gz')
    write_csv_gz(rates,ROOT_EVAL/'observed_hidden/expected_rates.csv.gz')
    write_csv_gz(source_future,ROOT_EVAL/'observed_source_future_expected_rates.csv.gz')
    write_csv_gz(tgt_obs,ROOT_EVAL/'heldout/public/observed_cells.csv.gz')
    write_csv_gz(tgt_counts,ROOT_EVAL/'heldout/public/counts.csv.gz')
    write_csv_gz(tgt_truth,ROOT_EVAL/'heldout/hidden/cell_truth.csv.gz')
    write_csv_gz(tgt_rates,ROOT_EVAL/'heldout/hidden/expected_rates.csv.gz')
    ROOT_MAN.mkdir(parents=True,exist_ok=True)
    write_csv_gz(probe_table,ROOT_MAN/'source_probe_cell_ids.csv.gz')
    np.savez_compressed(ROOT_EVAL/'observed_hidden/molecular_truth.npz',
                        q_true=result['q_true'], psi_z_true=result['psi_z_true'],
                        cell_id=source_truth.cell_id.to_numpy(str))

    # Numerical identity / Fisher orthogonality audit: purely for QC.
    nn = min(200,len(source_truth))
    at = source_truth.iloc[:nn]
    time_v = observed.loc[mask,'time'].to_numpy(float)[:nn]
    assert np.allclose(time_v,SOURCE_TIME)
    q,zgrad,_ = _material_derivatives(at.material_a_true.to_numpy(),
                                     at.u_true.to_numpy(),SOURCE_TIME,CFG,params)
    cos = np.abs(np.einsum('ig,ig->i',q,zgrad))/(
        np.maximum(np.linalg.norm(q,axis=1)*np.linalg.norm(zgrad,axis=1),1e-8))
    print('Fisher-orthogonality QC: median |cos(q,E_z)| =',float(np.median(cos)))
    assert float(np.quantile(cos,.95)) < 0.0005, 'Generator no longer Fisher orthogonal'

    # Manifest hashes exclude figures, which can be regenerated anytime.
    rels = [p.relative_to(ROOT).as_posix() for p in ROOT.rglob('*') if p.is_file()
            and 'quality_control/' not in p.relative_to(ROOT).as_posix()]
    rels = sorted(x for x in rels if x != 'manifests/dataset_manifest.json')
    hashes = {rel:file_hash(ROOT/rel) for rel in rels}
    payload = {
        'dataset_id':'synthetic_developmental_crescent_v1',
        'status':'frozen', 'generator_version':VERSION,
        'generator_origin':'Recovered original crescent v1.1 code embedded in prior stVCR benchmarking notebook; original geometry and expression model',
        'seed':NEW_SEED,'source_probe_seed':PROBE_SEED,
        'source_probe_fraction':PROBE_FRACTION,
        'source_t':SOURCE_TIME,'target_t':TARGET_TIME,
        'tissue_times':np.linspace(0,1,10).tolist(),
        'config':asdict(CFG),
        'simulator_hidden_truth_used_by_models':False,
        'coordinate_policy':'model sees raw observed x,y, per-section random translation retained; no rotations in original no-rotation variant',
        'target_separation':'independent target snapshot exclusively under evaluator_only',
        'n_observed_cells':len(observed),
        'n_source_t2_cells':len(source_ids),
        'n_source_probes':len(probe_ids),
        'n_heldout_cells':len(tgt_obs),
        'genes':GENE_COLS,
        'file_sha256':hashes
    }
    MANIFEST.write_text(json.dumps(payload,indent=2)+'\n',encoding='utf-8')
    print('DATASET FROZEN:',ROOT,flush=True)
    del result
    return payload


def load_locked_data():
    if not MANIFEST.is_file():
        return None
    d=json.loads(MANIFEST.read_text())
    if d['seed']!=NEW_SEED or d['config']!=asdict(CFG):
        raise RuntimeError('Dataset manifest exists, but simulator settings differ. Refusing reuse.')
    for rel,expected in d['file_sha256'].items():
        p=ROOT/rel
        if not p.is_file() or file_hash(p)!=expected:
            raise RuntimeError(f'Integrity verification failed: {p}')
    return d


manifest=load_locked_data()
if manifest is None:
    manifest=save_new_dataset()
else:
    print('REUSING frozen dataset; all file hashes verified:',ROOT)

# Load QC data ONLY AFTER the public/hidden split is committed to disk.
obs=pd.read_csv(ROOT_PUBLIC/'observed_cells.csv.gz')
count_df=pd.read_csv(ROOT_PUBLIC/'counts.csv.gz')
truth=pd.read_csv(ROOT_EVAL/'observed_hidden/cell_truth.csv.gz')
tgt_obs=pd.read_csv(ROOT_EVAL/'heldout/public/observed_cells.csv.gz')
tgt_truth=pd.read_csv(ROOT_EVAL/'heldout/hidden/cell_truth.csv.gz')

print('\nDataset summary:')
print('  Observed tissues:',obs.tissue_id.nunique(),'| observed cells:',len(obs))
print('  Source cells at 2/9:',manifest['n_source_t2_cells'],
      '| prespecified probe cells:',manifest['n_source_probes'])
print('  Independent target t=2.5/9:',len(tgt_obs),'cells')
print('  Genes:',len(GENE_COLS),'| rotations: OFF | translations: ON')
print('  Public model inputs:',ROOT_PUBLIC)
print('  Hidden evaluation only:',ROOT_EVAL)
print('  Manifest:',MANIFEST)